# 🧠 AI Workout Coach - ChromaDB Semantic Search Evaluation

This notebook evaluates the **hybrid retrieval search** (ChromaDB EphemeralClient + SQLite BLOBs + Google Gemini Embeddings + ChromaBm25EmbeddingFunction) and **Cross-Encoder reranking** system used in the AI Workout Coach database storage.

### 📈 Retrieval Methodology
1. **Semantic Search (ChromaDB EphemeralClient)**: Uses the `GoogleGeminiEmbeddingFunction` (model `gemini-embedding-001`) or falls back to local SentenceTransformer (`nomic-ai/nomic-embed-text-v1.5`) in-memory. Computes Cosine Similarity.
2. **Lexical Search (ChromaBm25EmbeddingFunction)**: Native BM25 keyword matching fallback.
3. **Hybrid Score Fusion**: Blends scores with a weighted ratio of **70% Semantic + 30% Keyword**.
4. **Threshold Filtering**: Only candidates with a combined hybrid score `>= 0.48` or high-confidence semantic matches `>= 0.60` are retained.
5. **Cross-Encoder Reranking**: The top 15 candidates are reranked using the `cross-encoder/ms-marco-MiniLM-L-6-v2` model (70% weight) combined with a metadata boost (30% weight) consisting of explicit subtag preferences (60%) and exponential recency decay (40%).

### 🔬 Evaluation Dataset Structure
- **50 Stored Memories**: Ground-truth user facts covering weekly schedules, body-composition assessments, injuries, dietary targets, and recovery habits.
- **200 Paraphrased Queries**: 4 diverse variations per memory (semantic paraphrasing, keyword emphasis, short queries) to evaluate query robustness.
- **50 Distractor Queries**: Unrelated general knowledge questions (capital cities, economics, coding, etc.) to verify threshold filter reliability (FPR).

In [ ]:
# 1. Initialize environments and import modules
import os
import sys
import time
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent workspace directory to sys.path to resolve imports correctly
notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

import database_chromadb as database
print("Embedding Model status:", database.get_db_status())

In [ ]:
# 2. Define the evaluation script data
# Storing references to memories and paraphrases
from test_semantic_search import TEST_MEMORIES, EVAL_QUERIES, DISTRACTOR_QUERIES
print(f"Loaded {len(TEST_MEMORIES)} ground-truth memories, {len(EVAL_QUERIES)} paraphrases, and {len(DISTRACTOR_QUERIES)} distractors.")

In [ ]:
# 3. Populate test database and index vectors
test_user = "eval_athlete_notebook_chroma"

# Clear out existing test athlete rows
conn = sqlite3.connect(database.DB_PATH)
c = conn.cursor()
c.execute("DELETE FROM memories WHERE username = ?", (test_user,))
conn.commit()
conn.close()

print("Populating memories...")
db_id_map = {}
from datetime import datetime, timedelta
now = datetime.now()
for idx, mem in enumerate(TEST_MEMORIES):
    hours_ago = (idx * 15) % 720
    timestamp = (now - timedelta(hours=hours_ago)).isoformat()
    database.save_memory(
        username=test_user,
        tag="semantic",
        query=mem["q"],
        response=mem["r"],
        subtag=mem["subtag"],
        timestamp=timestamp
    )
    
    conn = sqlite3.connect(database.DB_PATH)
    c = conn.cursor()
    c.execute("SELECT max(id) FROM memories WHERE username = ?", (test_user,))
    db_id_map[mem["id"]] = c.fetchone()[0]
    conn.close()

print(f"Populated evaluation database with {len(db_id_map)} records.")

In [ ]:
# 4. Run retrieval evaluation loop
print("Running paraphrased query tests...")
latencies = []
ranks = []
threshold_misses = 0

for idx, q_case in enumerate(EVAL_QUERIES):
    target_id = q_case["target_id"]
    tag = "semantic"
    query_text = q_case["query"]
    expected_id = db_id_map[target_id]
    
    t_start = time.time()
    results = database.vector_query_memories(test_user, tag, query_text, top_k=5)
    latencies.append((time.time() - t_start) * 1000.0)
    
    retrieved_ids = [r["id"] for r in results]
    if expected_id in retrieved_ids:
        ranks.append(retrieved_ids.index(expected_id) + 1)
    else:
        ranks.append(-1)
        if not results:
            threshold_misses += 1
            
print("Running distractor query tests...")
distractor_latencies = []
false_positives = 0
for dist_q in DISTRACTOR_QUERIES:
    t_start = time.time()
    res_sem = database.vector_query_memories(test_user, "semantic", dist_q, top_k=1)
    res_epi = database.vector_query_memories(test_user, "episodic", dist_q, top_k=1)
    res_pro = database.vector_query_memories(test_user, "procedural", dist_q, top_k=1)
    distractor_latencies.append((time.time() - t_start) * 1000.0)
    if res_sem or res_epi or res_pro:
        false_positives += 1

print("Evaluation loop complete.")

In [ ]:
# 5. Calculate and display metrics
total_queries = len(EVAL_QUERIES)
top_1 = sum(1 for r in ranks if r == 1)
top_3 = sum(1 for r in ranks if 1 <= r <= 3)
top_5 = sum(1 for r in ranks if 1 <= r <= 5)

acc_top1 = (top_1 / total_queries) * 100
acc_top3 = (top_3 / total_queries) * 100
acc_top5 = (top_5 / total_queries) * 100
fpr = (false_positives / len(DISTRACTOR_QUERIES)) * 100

all_latencies = latencies + distractor_latencies

print("===============================================")
print("SEMANTIC VECTOR SEARCH RETRIEVAL PERFORMANCE")
print("===============================================")
print(f"Top-1 Accuracy:  {acc_top1:.2f}% ({top_1}/{total_queries})")
print(f"Top-3 Accuracy:  {acc_top3:.2f}% ({top_3}/{total_queries})")
print(f"Top-5 Accuracy:  {acc_top5:.2f}% ({top_5}/{total_queries})")
print(f"Threshold Misses: {threshold_misses}/{total_queries}")
print(f"False Positives:  {false_positives}/{len(DISTRACTOR_QUERIES)} (FPR: {fpr:.2f}%)")
print("-----------------------------------------------")
print(f"Average Latency: {np.mean(all_latencies):.2f} ms")
print(f"Median Latency:  {np.median(all_latencies):.2f} ms")
print(f"95th Percentile: {np.percentile(all_latencies, 95):.2f} ms")
print(f"99th Percentile: {np.percentile(all_latencies, 99):.2f} ms")
print("===============================================")

In [ ]:
# 6. Plot performance distributions
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 5))

# Latency Distribution
plt.subplot(1, 2, 1)
sns.histplot(all_latencies, kde=True, bins=30, color="skyblue")
plt.axvline(np.mean(all_latencies), color="red", linestyle="--", label=f"Mean: {np.mean(all_latencies):.1f}ms")
plt.axvline(np.percentile(all_latencies, 95), color="orange", linestyle="-.", label=f"p95: {np.percentile(all_latencies, 95):.1f}ms")
plt.title("Query Latency Distribution (ms)")
plt.xlabel("Latency (ms)")
plt.ylabel("Count")
plt.legend()

# Retrieval Accuracy by Rank
plt.subplot(1, 2, 2)
categories = ['Top-1', 'Top-3', 'Top-5', 'Miss/Filter']
counts = [
    top_1,
    top_3 - top_1,
    top_5 - top_3,
    total_queries - top_5
]
colors = ["#2ecc71", "#3498db", "#9b59b6", "#e74c3c"]
plt.bar(categories, counts, color=colors)
plt.title("Retrieval Accuracy by Rank Placement")
plt.xlabel("Retrieved Rank")
plt.ylabel("Query Count")
for i, val in enumerate(counts):
    pct = (val / total_queries) * 100
    plt.text(i, val + 2, f"{pct:.1f}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Diagnostic Search Reports
Run diagnostic search query tests to print score breakdowns.

In [ ]:
from query_test import run_debug_search

# 1. Choose your query and tag
my_query = "Do i have any pain in body?"
my_tag = "semantic"  # Try: "semantic", "episodic", or "procedural"

# 2. Run the diagnostic search (using the athlete user populated in Cell 3)
run_debug_search("eval_athlete_notebook_chroma", my_tag, my_query)

In [ ]:
from query_test import run_debug_search

# 1. Type a query that matches onboarding/explicit memories 
my_query = "I will have vegan food only"
my_tag = "semantic"  

# 2. Run the diagnostic search
run_debug_search("eval_athlete_notebook_chroma", my_tag, my_query)